# ResNet‑18 Feature Maps and Image Embedding (Explained)

This notebook demonstrates **how ResNet‑18 transforms an image** step‑by‑step.

It focuses on:
- what feature maps represent at each layer
- how visual abstraction evolves using 3×3 kernels (stride 1)
- how a final 512‑D image embedding is produced and exported


## 1. Imports
Libraries for deep learning, visualization, and data export.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from torchvision import models, transforms
from PIL import Image
from pathlib import Path


## 2. Device selection
CPU is used to ensure compatibility with Binder and GitHub.

In [ ]:
device = torch.device("cpu")
print("Using device:", device)

## 3. Load ResNet‑18 in embedding mode

The final classification layer is removed so the network outputs a **512‑dimensional embedding**.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Identity()
model.eval()
print("ResNet‑18 loaded in embedding mode")

## 4. Store feature maps using hooks

A **feature map** is the activation of a learned filter.
Each channel answers:

> *Where does my learned pattern occur, and how strongly?*

In [ ]:
feature_maps = {}

def hook_fn(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach().cpu()
    return hook

model.conv1.register_forward_hook(hook_fn("conv1"))
model.layer1.register_forward_hook(hook_fn("layer1"))
model.layer2.register_forward_hook(hook_fn("layer2"))
model.layer3.register_forward_hook(hook_fn("layer3"))
model.layer4.register_forward_hook(hook_fn("layer4"))

## 5. Image preprocessing (ImageNet standard)
3×3 convolutions expect consistent scaling and normalization.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

## 6. Load input image (from GitHub data folder)

In [ ]:
img_path = Path("data") / "25DIV3FAY_1002.jpg"
assert img_path.exists(), "Image not found in data/ folder"

img = Image.open(img_path).convert("RGB")
x = transform(img).unsqueeze(0)

plt.figure(figsize=(4,4))
plt.imshow(img)
plt.axis("off")
plt.title("Input Image")
plt.show()

## 7. Forward pass
This computes feature maps at **all convolutional layers**.

In [ ]:
with torch.no_grad():
    embedding = model(x).squeeze().numpy()

print("Embedding shape:", embedding.shape)

## 8. Feature‑map visualization utility

In [ ]:
def visualize_feature_maps(fmap, title, max_channels=16):
    fmap = fmap.squeeze(0)
    n = min(fmap.shape[0], max_channels)
    grid = math.ceil(math.sqrt(n))
    plt.figure(figsize=(8,8))
    for i in range(n):
        fm = fmap[i]
        fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-6)
        plt.subplot(grid, grid, i+1)
        plt.imshow(fm, cmap='viridis')
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## Visualization: conv1 — edges and colors

- Learns edge orientation
- Color and intensity gradients
- Very similar to classical edge detectors

In [ ]:
visualize_feature_maps(feature_maps['conv1'], 'conv1: edges and color gradients')

## Visualization: layer1 — low‑level textures

- Combines edges using 3×3 stride‑1 convolutions
- Detects repetitive micro‑patterns
- Texture sensitivity emerges

In [ ]:
visualize_feature_maps(feature_maps['layer1'], 'layer1: low-level textures')

## Visualization: layer2 — canopy patterns / parts

- Groups textures into mid‑level regions
- Responds to vegetation clusters and object parts

In [ ]:
visualize_feature_maps(feature_maps['layer2'], 'layer2: canopy patterns and parts')

## Visualization: layer3 — spatial structure

- Captures layout and spacing
- Structural abstraction dominates

In [ ]:
visualize_feature_maps(feature_maps['layer3'], 'layer3: spatial structure')

## Visualization: layer4 — high‑level semantics

- Class‑relevant semantic concepts
- No longer visually interpretable (expected)

In [ ]:
visualize_feature_maps(feature_maps['layer4'], 'layer4: high-level semantics')

## 9. Embedding DataFrame and CSV export
The embedding encodes **concept strength**, not spatial location.

In [ ]:
df_embedding = pd.DataFrame([embedding], columns=[f'emb_{i}' for i in range(512)])
df_embedding.insert(0, 'image_id', img_path.name)
df_embedding

In [ ]:
df_embedding.to_csv('image_embedding.csv', index=False)
print('Embedding saved to image_embedding.csv')